# Módulo 2 · Clase 3 (Teoría) — De las CNN a los Vision Transformers
### Deep Learning · Apunte de cátedra

> **Cómo usar este notebook (profesor):** apunte vivo. Markdown para explicar, código para demos en vivo, imágenes = slides embebidas por URL desde el repo.

**Objetivos.** Al terminar, los estudiantes deberían poder:
1. Explicar por qué un MLP no es adecuado para imágenes y qué aporta la convolución (localidad, pesos compartidos, invariancia a traslación).
2. Describir las piezas de una CNN: capas convolucionales, *feature maps*, stride, padding y pooling.
3. Reconocer las arquitecturas clásicas (LeNet → AlexNet → VGG → ResNet) y la idea de *skip connections*.
4. Entender el transfer learning: extracción de características vs. fine-tuning.
5. Explicar la idea de los Vision Transformers (parches como tokens) y compararlos con las CNN.

**Agenda (≈ 3 horas, con un descanso):**
| Bloque | Tema | ~min |
|---|---|---|
| 0 | ¿Por qué un MLP no basta para imágenes? | 15 |
| 1 | La convolución | 30 |
| 2 | La capa convolucional y el pooling | 30 |
| 3 | Anatomía de una CNN | 20 |
| — | *Descanso* | 10 |
| 4 | Arquitecturas clásicas | 30 |
| 5 | Transfer learning | 25 |
| 6 | Vision Transformers | 30 |
| 7 | Más allá de la clasificación | 10 |


In [ ]:
# Setup
import numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); np.random.seed(0)
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available())

También instalamos nuestra librería para visualizaciones y demos

In [ ]:
!pip install dlviz

## Bloque 0 · ¿Por qué un MLP no basta para imágenes?

En el Módulo 1 aplanábamos la imagen a un vector y la pasábamos por un MLP. Eso tiene dos problemas graves:

1. **Demasiados parámetros.** Una imagen de 224×224×3 aplanada son 150.528 entradas. Conectarla a una sola capa de 1000 neuronas son ~150 **millones** de pesos... en una sola capa.
2. **Ignora la estructura espacial.** Al aplanar, perdemos la noción de *vecindad*: dos píxeles contiguos quedan tan "lejos" como dos píxeles en esquinas opuestas. La información visual es **local** (bordes, texturas) y eso se desperdicia.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m2_00_mlp_imagen.png" width="760">

<sub>Un MLP sobre la imagen aplanada: conexiones densas, explosión de parámetros y pérdida de la estructura espacial.</sub>

In [ ]:
# Demo: la explosión de parámetros del MLP vs una capa convolucional
H, W, C = 224, 224, 3
mlp_params  = H*W*C * 1000          # primera capa densa a 1000 neuronas
conv_params = (3*3*C)*64 + 64       # capa conv 3x3, 3->64 canales
print(f"MLP  (primera capa densa, 1000 unidades): {mlp_params:,} parámetros")
print(f"Conv (3x3, 3->64 canales):                {conv_params:,} parámetros")
print(f"\nLa capa conv usa {mlp_params/conv_params:,.0f}x menos parámetros.")

## Bloque 1 · La convolución

La convolución desliza un **kernel** (filtro) pequeño sobre la imagen y, en cada posición, calcula un producto punto entre el kernel y el parche local. El resultado es un nuevo mapa que resalta cierto patrón.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m2_01_convolucion.png" width="760">

<sub>Convolución: un kernel recorre la imagen y produce, en cada posición, un valor a partir del parche local.</sub>

Tres propiedades la hacen ideal para imágenes:
- **Localidad:** cada salida depende solo de un vecindario pequeño.
- **Pesos compartidos:** el *mismo* kernel se aplica en toda la imagen → pocos parámetros y detección del patrón sin importar *dónde* aparezca.
- **Invariancia a traslación:** si el patrón se mueve, la respuesta se mueve con él.

Dependiendo de sus pesos, un kernel detecta bordes, texturas, etc. **Demo:** apliquemos kernels de detección de bordes a una imagen sintética.

In [ ]:
from dlviz import conv_interactiva
conv_interactiva()

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m2_02_conv_bordes.png" width="760">

<sub>Un kernel adecuado actúa como detector de bordes: la respuesta resalta los contornos de la imagen.</sub>

In [ ]:
# Demo: la convolución como detector de bordes
# Imagen sintética con figuras
img = np.zeros((64, 64), dtype=np.float32)
img[12:52, 12:24] = 1.0          # rectángulo
img[20:44, 36:56] = 1.0          # cuadrado
x = torch.tensor(img).view(1,1,64,64)

# Kernels Sobel (bordes verticales y horizontales)
sobel_x = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
sobel_y = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)
edge_x = F.conv2d(x, sobel_x, padding=1)
edge_y = F.conv2d(x, sobel_y, padding=1)

fig, ax = plt.subplots(1,3, figsize=(11,3.5))
ax[0].imshow(img, cmap='gray');                 ax[0].set_title("original")
ax[1].imshow(edge_x[0,0].abs(), cmap='gray');   ax[1].set_title("bordes verticales (Sobel-x)")
ax[2].imshow(edge_y[0,0].abs(), cmap='gray');   ax[2].set_title("bordes horizontales (Sobel-y)")
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

## Bloque 2 · La capa convolucional y el pooling

Una **capa convolucional** no usa un solo kernel sino un **banco de kernels**. Cada kernel produce un *feature map*; apilados forman un **volumen** de salida cuya profundidad es el número de kernels.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m2_03_capa_conv.png" width="760">

<sub>Una capa conv toma un volumen de entrada (p. ej. 64×64×3), aplica varios kernels y produce un volumen de feature maps (64×64×8).</sub>

**Hiperparámetros:** número de kernels (profundidad de salida), tamaño del kernel ($K$), **stride** ($S$, cuánto se desplaza), y **padding** ($P$, relleno de borde). El tamaño de salida es:

$$ W_{out} = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1 $$

In [ ]:
# Demo: fórmula del tamaño de salida de una convolución
def conv_out(W, K, P, S):
    return (W - K + 2*P)//S + 1
for (W,K,P,S) in [(32,3,1,1),(32,3,0,1),(32,5,2,1),(32,3,1,2)]:
    print(f"entrada {W}, kernel {K}, padding {P}, stride {S}  ->  salida {conv_out(W,K,P,S)}")

In [ ]:
from dlviz import pad_stride_interactiva
pad_stride_interactiva()

El **pooling** reduce la resolución espacial (típicamente *max pooling* 2×2, stride 2): resume cada región por su valor máximo. Reduce cómputo y da algo de robustez a pequeñas traslaciones.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m2_04_pooling.png" width="680">

<sub>Max pooling 2×2: cada bloque se resume por su máximo, reduciendo a la mitad alto y ancho.</sub>

In [ ]:
# Demo: max pooling 2x2
a = torch.tensor([[1.,2,5,6],[3,4,7,8],[9,10,13,14],[11,12,15,16]]).view(1,1,4,4)
print("entrada 4x4:\n", a[0,0])
print("\nmax-pool 2x2 ->\n", F.max_pool2d(a, 2)[0,0])

In [ ]:
from dlviz import pooling_interactiva
pooling_interactiva()

## Bloque 3 · Anatomía de una CNN

Una CNN típica alterna **conv → activación → pooling** varias veces (extrae características cada vez más abstractas) y termina con capas densas para clasificar. Las primeras capas detectan bordes; las profundas, partes y objetos.

**Demo:** una CNN pequeña y su conteo de parámetros, comparado con un MLP equivalente.

In [ ]:
# Demo: una CNN pequeña (para imágenes 3x32x32, estilo CIFAR-10)
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 32x16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 64x8x8
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(64*8*8, n_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

cnn = SmallCNN()
x = torch.randn(1,3,32,32)
print("salida:", cnn(x).shape)
cnn_params = sum(p.numel() for p in cnn.parameters())
mlp_params = 3*32*32*512 + 512*10   # MLP comparable: 3072->512->10
print(f"\nParámetros CNN:  {cnn_params:,}")
print(f"Parámetros MLP (3072->512->10): {mlp_params:,}")

In [ ]:
from dlviz import multicanal_interactiva
multicanal_interactiva()

## Bloque 4 · Arquitecturas clásicas

La historia del deep learning en visión es una carrera por redes más profundas y efectivas.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m2_05_lenet.png" width="760">

<sub>LeNet-5 (1998): la CNN pionera, para reconocer dígitos. conv → pooling → conv → pooling → capas densas.</sub>

- **LeNet-5 (1998):** la primera CNN exitosa (dígitos). [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M2/01_LeNet.png)
- **AlexNet (2012):** ganó ImageNet por amplio margen y desató la era del deep learning. ReLU, dropout, entrenamiento en GPU. [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M2/02_AlexNet.png)
- **VGG (2014):** demostró que apilar muchos kernels pequeños (3×3) funciona muy bien; arquitectura simple y uniforme. [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M2/03_VGGNet.png)
- **GoogLeNet/Inception (2014):** módulos con kernels de varios tamaños en paralelo. [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M2/04_InceptionNet.png)
- **ResNet (2015):** permitió entrenar redes de 100+ capas gracias a las *skip connections*. [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M2/05_ResNet.png)

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m2_06_revolution_depth.png" width="760">

<sub>La 'revolución de la profundidad' en ImageNet: de 8 capas (AlexNet) a 152 (ResNet), con caída sostenida del error.</sub>

### El problema de la profundidad y las *skip connections*
Apilar más capas debería ayudar, pero en la práctica las redes muy profundas se volvían **más difíciles de entrenar** (los gradientes se degradan). **ResNet** lo resolvió con conexiones residuales: la capa aprende un *residuo* $F(x)$ y la salida es $F(x) + x$. El atajo deja pasar el gradiente sin atenuarse.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m2_07_resnet_residual.png" width="620">

<sub>Bloque residual: la entrada x se suma a la salida F(x). El atajo (identidad) facilita el flujo del gradiente en redes profundas.</sub>

## Bloque 5 · Transfer learning ([Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M2/06_Transfer_Learning.png)) 

Entrenar una CNN grande desde cero requiere millones de imágenes y mucho cómputo. **Transfer learning** reutiliza una red ya entrenada en un dataset enorme (ImageNet) y la adapta a *nuestra* tarea, que suele tener pocos datos.

¿Por qué funciona? Porque las primeras capas aprenden características **genéricas** (bordes, texturas) que sirven para casi cualquier imagen; solo las últimas son específicas de la tarea original.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m2_08_feature_hierarchy.png" width="760">

<sub>Jerarquía de características aprendidas: bordes (bajo nivel) → partes (nivel medio) → objetos (alto nivel). Las primeras son reutilizables.</sub>

Dos estrategias:
- **Feature extraction:** congelar la red preentrenada (usarla como extractor de características fijo) y entrenar solo un nuevo clasificador encima. Ideal con pocos datos.
- **Fine-tuning:** continuar entrenando (con learning rate bajo) algunas o todas las capas preentrenadas, ajustándolas a la nueva tarea. Mejor cuando hay más datos.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m2_09_transfer_learning.png" width="760">

<sub>Transfer learning: el modelo preentrenado actúa como extractor de características; se entrena un nuevo clasificador encima.</sub>

## Bloque 6 · Vision Transformers (ViT) ([Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M2/07_Vision_Transformer.png))

En 2020 los Transformers —dominantes en NLP— llegaron a la visión. La idea es sorprendentemente directa: **tratar una imagen como una secuencia de parches**, igual que una frase es una secuencia de palabras.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m2_11_vit_parches.png" width="520">

<sub>Primer paso de un ViT: la imagen se divide en parches (p. ej. 16×16 px). Cada parche será un 'token'.</sub>

El pipeline de un ViT:
1. **Dividir** la imagen en $N$ parches (por ejemplo de 16×16 px).
2. **Proyectar** cada parche a un vector de dimensión $D$ (embedding lineal).
3. **Sumar** un *positional embedding* (el Transformer no conoce el orden por sí solo).
4. Agregar un **token de clasificación** ([CLS]) extra.
5. Pasar todo por un **Transformer estándar** (el mismo de NLP: self-attention + MLP).
6. Usar la salida del token [CLS] para **clasificar**.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m2_10_vit_arquitectura.png" width="760">

<sub>Arquitectura ViT: parches → proyección a D dimensiones + positional embedding → Transformer → el token de clasificación se proyecta a scores de clase.</sub>

### ViT vs. CNN
- Las **CNN** tienen un *sesgo inductivo* fuerte (localidad, invariancia a traslación), lo que las hace eficientes con **pocos datos**.
- Los **ViT** no tienen ese sesgo: deben *aprenderlo* de los datos, así que necesitan **datasets muy grandes** (o preentrenamiento) para brillar — pero entonces escalan mejor y capturan relaciones de largo alcance.

En la práctica hoy se usan ambos, y arquitecturas híbridas. Esto conecta directamente con el Módulo 3, donde veremos el mecanismo de atención en detalle.

## Bloque 7 · Más allá de la clasificación

La clasificación (¿qué hay en la imagen?) es solo una tarea. La visión computacional moderna incluye:
- **Detección de objetos:** *qué* y *dónde* (cajas). Familias R-CNN, **YOLO**, SSD, RetinaNet. [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M2/08_YOLO.png)
- **Segmentación:** etiquetar *cada píxel* (semántica) o cada instancia (Mask R-CNN). [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M2/09_Vision_con_ViT.png)

Estas tareas reutilizan los *backbones* (CNN o ViT) que vimos hoy como extractores de características. En el repositorio del curso hay slides y labs dedicados (`Labs/` con YOLO, SSD, Mask R-CNN) para profundizar.

## Cierre
Vimos por qué la convolución es la operación natural para imágenes, cómo se arma una CNN, la evolución de las arquitecturas hasta ResNet, el transfer learning y el salto a los Vision Transformers.

**En la clase práctica (Clase 4)** van a entrenar una CNN en CIFAR-10, aplicar data augmentation y hacer transfer learning con una ResNet preentrenada y un ViT.